# 07. 순환 신경망 (Recurrent Neural Network)

이번 장에서는 **시퀀스(Sequence)** 데이터를 다루기 위한 **순환 신경망(Recurrent Neural Network, RNN)**을 정리한다. 앞에서 배운 신경망들은 입력이 출력층 방향으로만 흐르는 **피드 포워드 신경망(Feed Forward Neural Network)**이었다. RNN은 은닉층의 출력을 다시 자기 자신의 다음 입력으로 되돌려 보내면서, 순서가 있는 데이터를 처리한다.

> **이 장의 흐름** : RNN의 구조와 수식 → Numpy로 직접 구현 → `nn.RNN()` 사용법 → 깊은 RNN·양방향 RNN → 바닐라 RNN의 한계(장기 의존성) → 이를 보완한 LSTM·GRU → 문자 단위 RNN(Char RNN) 실습.
>
> 관통하는 핵심은 하나다. **현재 시점의 출력은 현재 입력뿐 아니라 과거 시점의 계산 결과(은닉 상태)에도 의존한다.** 이 되먹임 구조가 RNN을 시퀀스 모델로 만든다.

# 1. 순환 신경망 (Recurrent Neural Network, RNN)

RNN은 입력과 출력을 **시퀀스 단위**로 처리하는 모델이다. 예를 들어 번역기는 번역할 문장(단어 시퀀스)을 입력받아 번역된 문장(단어 시퀀스)을 출력한다. 이렇게 순서가 있는 데이터를 다루기 위해 고안된 모델을 **시퀀스 모델**이라 하며, RNN은 그중 딥 러닝에서 가장 기본적인 모델이다.

> 이름이 비슷한 **재귀 신경망(Recursive Neural Network)**과 순환 신경망(Recurrent Neural Network)은 전혀 다른 개념이므로 혼동하지 않는다.

## RNN의 구조 : 셀과 은닉 상태

앞서 배운 신경망들은 은닉층에서 활성화 함수를 지난 값이 **오직 출력층 방향으로만** 향했다. 이런 신경망을 **피드 포워드 신경망**이라 한다. 반면 RNN은 은닉층 노드의 출력을 출력층으로 보내는 동시에, **다시 자기 자신의 다음 계산을 위한 입력으로 되돌려 보낸다.**

RNN에서 은닉층의 결과를 내보내는 노드를 **셀(cell)**이라 부른다. 이 셀은 이전의 값을 기억하려는 메모리 역할을 하므로 **메모리 셀** 또는 **RNN 셀**이라고도 한다. 메모리 셀은 각 **시점(time step)**마다 바로 이전 시점에서 자신이 내보낸 값을 다시 입력으로 사용하는 재귀적 활동을 한다.

이때 메모리 셀이 출력층 방향으로, 또는 다음 시점 $t+1$ 의 자기 자신에게 보내는 값을 **은닉 상태(hidden state)**라고 한다. 다시 말해 $t$ 시점의 메모리 셀은 $t-1$ 시점의 메모리 셀이 보낸 은닉 상태값을 $t$ 시점의 은닉 상태 계산을 위한 입력으로 사용한다.

> RNN은 사이클을 그리는 화살표로 재귀 형태로 표현하기도 하고, 시점의 흐름에 따라 여러 개로 펼쳐서 표현하기도 한다. 두 그림은 완전히 동일한 RNN을 나타낸다. 또한 피드 포워드 신경망에서는 '뉴런' 단위를 썼지만, RNN에서는 입력층·출력층은 **입력 벡터·출력 벡터**, 은닉층은 **은닉 상태**라는 표현을 주로 쓴다.

## RNN의 다양한 형태

RNN은 입력과 출력의 길이를 다르게 설계할 수 있어 다양한 용도로 쓰인다. 각 시점의 입출력 단위는 사용자가 정하기 나름이지만, 자연어 처리에서 가장 보편적인 단위는 **단어 벡터**다.

- **일 대 다 (one-to-many)** : 하나의 입력에 여러 개의 출력. 하나의 이미지를 입력받아 사진 제목(단어들의 시퀀스)을 출력하는 **이미지 캡셔닝(Image Captioning)**에 쓸 수 있다.
- **다 대 일 (many-to-one)** : 단어 시퀀스를 입력받아 하나의 출력. 입력 문서가 긍정인지 부정인지 판별하는 **감성 분류(sentiment classification)**, 메일이 정상인지 스팸인지 판별하는 **스팸 메일 분류**에 쓸 수 있다.
- **다 대 다 (many-to-many)** : 시퀀스를 입력받아 시퀀스를 출력. 입력 문장으로부터 대답을 출력하는 **챗봇**, 번역된 문장을 출력하는 **번역기**, **개체명 인식**, **품사 태깅** 등이 이에 속한다.

## RNN의 수식

현재 시점 $t$ 의 은닉 상태값을 $h_t$ 라 하자. 은닉층의 메모리 셀은 $h_t$ 를 계산하기 위해 두 개의 가중치를 갖는다. 하나는 입력값 $x_t$ 를 위한 가중치 $W_x$, 다른 하나는 이전 시점의 은닉 상태값 $h_{t-1}$ 를 위한 가중치 $W_h$ 다.

- **은닉층** : $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$
- **출력층** : $y_t = f(W_y h_t + b)$ &nbsp; (단, $f$ 는 비선형 활성화 함수 중 하나)

은닉 상태 $h_t$ 의 활성화 함수로는 주로 **하이퍼볼릭탄젠트(tanh)**가 쓰이며, ReLU로 바꿔 쓰기도 한다. 출력층 $y_t$ 의 활성화 함수는 문제에 따라 다른데, 이진 분류면 시그모이드, 다중 클래스 분류면 소프트맥스를 쓴다.

RNN의 은닉층 연산은 벡터·행렬 연산으로 이해할 수 있다. 단어 벡터의 차원을 $d$, 은닉 상태의 크기를 $D_h$ 라 할 때 각 벡터·행렬의 크기는 다음과 같다.

$$ x_t : (d \times 1) \quad W_x : (D_h \times d) \quad W_h : (D_h \times D_h) \quad h_{t-1} : (D_h \times 1) \quad b : (D_h \times 1) $$

> 각 가중치 $W_x, W_h, W_y$ 의 값은 **모든 시점에서 동일하게 공유**된다. 단, 은닉층이 2개 이상이면 각 은닉층의 가중치는 서로 다르다.

# 2. 파이썬으로 RNN 구현하기

`nn.RNN()`을 쓰기 전에, 메모리 셀이 은닉 상태를 계산하는 과정을 **Numpy로 직접 구현**하며 동작 원리를 확인한다. 앞서 정의한 은닉 상태 식은 다음과 같다.

$$ h_t = \tanh(W_x x_t + W_h h_{t-1} + b) $$

의사 코드(pseudocode)로 개념을 정리하면, 각 시점마다 입력 `input_t`와 이전 은닉 상태 `hidden_state_t`를 받아 tanh를 통과시켜 현재 은닉 상태를 계산하고, 그 결과를 다시 다음 시점의 은닉 상태로 넘기는 반복이다.

> 아래 실습은 이해를 돕기 위해 `(timesteps, input_size)` 크기의 2D 텐서를 입력으로 가정한다. 다만 실제 파이토치에서는 `(batch_size, timesteps, input_size)` 크기의 **3D 텐서**를 입력으로 받는다는 점을 기억한다.

In [ ]:
import numpy as np

timesteps = 10    # 시점의 수. NLP에서는 보통 문장의 길이가 된다.
input_size = 4    # 입력의 차원. NLP에서는 보통 단어 벡터의 차원이 된다.
hidden_size = 8   # 은닉 상태의 크기. 메모리 셀의 용량이다.

# 입력에 해당되는 2D 텐서
inputs = np.random.random((timesteps, input_size))

# 초기 은닉 상태는 0 벡터로 초기화
hidden_state_t = np.zeros((hidden_size,))

print(hidden_state_t)  # 모든 차원이 0인 초기 은닉 상태

가중치와 편향을 각 크기에 맞게 정의한다. $W_x$ 는 (은닉 상태의 크기 × 입력의 차원), $W_h$ 는 (은닉 상태의 크기 × 은닉 상태의 크기), $b$ 는 (은닉 상태의 크기)의 크기를 가진다.

In [ ]:
Wx = np.random.random((hidden_size, input_size))   # (8, 4) 입력에 대한 가중치
Wh = np.random.random((hidden_size, hidden_size))  # (8, 8) 은닉 상태에 대한 가중치
b = np.random.random((hidden_size,))               # (8,)   편향(bias)

print(np.shape(Wx))
print(np.shape(Wh))
print(np.shape(b))

이제 모든 시점의 은닉 상태를 출력한다고 가정하고 RNN 층을 동작시킨다. 각 시점마다 $W_x x_t + W_h h_{t-1} + b$ 를 계산해 tanh를 통과시키고, 그 결과를 다음 시점의 은닉 상태로 넘긴다.

In [ ]:
total_hidden_states = []

# 메모리 셀 동작
for input_t in inputs:  # 각 시점에 따라서 입력값이 입력됨
    # Wx * Xt + Wh * Ht-1 + b
    output_t = np.tanh(np.dot(Wx, input_t) + np.dot(Wh, hidden_state_t) + b)
    total_hidden_states.append(list(output_t))  # 각 시점의 은닉 상태를 축적
    hidden_state_t = output_t                   # 현재 은닉 상태를 다음 시점으로 전달

total_hidden_states = np.stack(total_hidden_states, axis=0)

# (timesteps, output_dim) = (10, 8) 크기의 은닉 상태 2D 텐서
print(total_hidden_states.shape)
print(total_hidden_states)

각 시점 $t$ 별 메모리 셀의 출력이 `(timesteps, output_dim)`, 즉 `(10, 8)` 크기로 쌓인다.

# 3. 파이토치의 nn.RNN()

파이토치에서는 `nn.RNN()`으로 RNN 셀을 간단히 구현한다. 인자로 입력의 크기와 은닉 상태의 크기를 넘기고, `batch_first=True`를 주어 입력 텐서의 첫 번째 차원이 배치 크기임을 알린다. 입력 텐서는 `(배치 크기 × 시점의 수 × 매 시점의 입력)` 크기를 가진다.

In [ ]:
import torch
import torch.nn as nn

input_size = 5   # 매 시점마다 들어가는 입력의 크기
hidden_size = 8  # 은닉 상태의 크기 (대표적인 RNN 하이퍼파라미터)

# (batch_size, time_steps, input_size)
# 배치 크기 1, 10번의 시점 동안 5차원 입력 벡터가 들어가는 텐서
inputs = torch.Tensor(1, 10, 5)

cell = nn.RNN(input_size, hidden_size, batch_first=True)
outputs, _status = cell(inputs)

`nn.RNN()`은 두 개의 값을 리턴한다. 첫 번째는 **모든 시점(timesteps)의 은닉 상태**, 두 번째는 **마지막 시점의 은닉 상태**다.

In [ ]:
print(outputs.shape)   # 모든 time-step의 hidden_state
print(_status.shape)   # 최종 time-step의 hidden_state

첫 번째 리턴값은 `(1, 10, 8)` 크기로, 10번의 시점 동안 8차원의 은닉 상태가 출력되었다는 의미다. 두 번째 리턴값인 마지막 시점의 은닉 상태는 `(1, 1, 8)` 크기를 가진다.

# 4. 깊은 순환 신경망 (Deep Recurrent Neural Network)

RNN도 은닉층을 여러 개 쌓을 수 있다. 은닉층이 2개인 깊은 RNN에서는 첫 번째 은닉층이 모든 시점의 은닉 상태값을 다음 은닉층으로 보내준다. 파이토치에서는 `nn.RNN()`의 인자 `num_layers`에 값을 전달해 층을 쌓는다.

In [ ]:
# (batch_size, time_steps, input_size)
inputs = torch.Tensor(1, 10, 5)

cell = nn.RNN(input_size=5, hidden_size=8, num_layers=2, batch_first=True)
outputs, _status = cell(inputs)

print(outputs.shape)   # 모든 time-step의 hidden_state
print(_status.shape)   # (층의 개수, 배치 크기, 은닉 상태의 크기)

첫 번째 리턴값의 크기 `(1, 10, 8)`은 층이 1개였을 때와 달라지지 않는다. 이는 **마지막 층**의 모든 시점 은닉 상태이기 때문이다. 반면 두 번째 리턴값은 `(층의 개수, 배치 크기, 은닉 상태의 크기)` = `(2, 1, 8)`로 달라진다.

# 5. 양방향 순환 신경망 (Bidirectional Recurrent Neural Network)

양방향 RNN은 시점 $t$ 의 출력을 예측할 때 **이전 시점의 데이터뿐 아니라 이후 시점의 데이터도** 활용한다는 아이디어에 기반한다.

> 영어 빈칸 채우기를 떠올려 보자. *Exercise is very effective at [ ] belly fat.* 에서 정답 `reducing`을 맞히려면 앞 단어들만으로는 부족하고, 뒤에 오는 목적어 `belly fat`을 알아야 한다. 이렇듯 정답의 힌트가 미래 시점에 있는 경우가 많아, 앞뒤 양방향의 정보를 모두 쓰기 위해 고안된 것이 양방향 RNN이다.

양방향 RNN은 하나의 출력을 위해 **두 개의 메모리 셀**을 쓴다. 첫 번째 셀은 앞 시점의 은닉 상태(Forward States)를, 두 번째 셀은 뒤 시점의 은닉 상태(Backward States)를 전달받아 계산하며, 두 값 모두가 출력층에서 사용된다. 파이토치에서는 `nn.RNN()`의 인자 `bidirectional=True`로 구현한다.

In [ ]:
# (batch_size, time_steps, input_size)
inputs = torch.Tensor(1, 10, 5)

cell = nn.RNN(input_size=5, hidden_size=8, num_layers=2,
              batch_first=True, bidirectional=True)
outputs, _status = cell(inputs)

print(outputs.shape)   # (배치 크기, 시퀀스 길이, 은닉 상태의 크기 x 2)
print(_status.shape)   # (층의 개수 x 2, 배치 크기, 은닉 상태의 크기)

첫 번째 리턴값은 `(1, 10, 16)`으로, 은닉 상태의 크기가 단방향일 때의 **2배**가 되었다. 양방향의 은닉 상태 값들이 연결(concatenate)되었기 때문이다. 두 번째 리턴값은 `(4, 1, 8)`로, 정방향 기준 마지막 시점이자 역방향 기준 첫 시점의 출력값을 층의 개수만큼 쌓은 결과다.

> 다른 신경망과 마찬가지로 은닉층을 무조건 추가한다고 성능이 좋아지는 것은 아니다. 층을 늘리면 학습 가능한 양이 많아지지만, 그만큼 훈련 데이터도 더 많이 필요하다.

# 6. 바닐라 RNN의 한계 : 장기 의존성 문제

지금까지 배운 가장 단순한 형태의 RNN을 **바닐라 RNN(Vanilla RNN)**이라 한다. 앞으로 LSTM과 비교하며 'RNN'이라 하면 모두 이 바닐라 RNN을 가리킨다.

바닐라 RNN은 비교적 **짧은 시퀀스**에서만 효과를 보인다. 시점(time step)이 길어질수록 앞쪽의 정보가 뒤로 충분히 전달되지 못하고 점점 손실되기 때문이다. 시점이 충분히 길면 맨 앞 입력의 영향력은 거의 의미가 없어질 수도 있다.

> 예를 들어 *"모스크바에 여행을 왔는데 … 직장 상사한테 전화가 왔어. 어디냐고 묻더라구. 그래서 나는 말했지. 저 여행 왔는데요. 여기 ___"* 의 다음 단어를 예측하려면 장소 정보인 '모스크바'가 필요하다. 그런데 이 단어는 문장의 맨 앞에 있어, RNN이 충분한 기억력을 갖지 못하면 엉뚱한 예측을 한다. 이를 **장기 의존성 문제(the problem of Long-Term Dependencies)**라고 한다.

# 7. LSTM (Long Short-Term Memory)

**장단기 메모리(Long Short-Term Memory, LSTM)**는 바닐라 RNN의 장기 의존성 문제를 보완한 RNN의 일종이다. 은닉층의 메모리 셀에 **입력 게이트·삭제 게이트·출력 게이트**를 추가해 불필요한 기억은 지우고 기억해야 할 것은 정한다.

LSTM은 은닉 상태(hidden state)를 계산하는 식이 바닐라 RNN보다 복잡해졌고, **셀 상태(cell state)** $C_t$ 라는 값이 추가되었다. 셀 상태 또한 은닉 상태처럼 이전 시점의 값이 다음 시점 계산의 입력으로 쓰인다. LSTM은 RNN보다 긴 시퀀스 입력을 처리하는 데 탁월하다.

## 바닐라 RNN 내부 다시 보기

LSTM을 이해하기 전에 바닐라 RNN의 내부를 짚는다. 바닐라 RNN은 $x_t$ 와 $h_{t-1}$ 두 입력이 각각의 가중치와 곱해져 메모리 셀의 입력이 되고, 이를 하이퍼볼릭탄젠트 함수에 통과시킨 값이 은닉 상태가 된다.

$$ h_t = \tanh(W_x x_t + W_h h_{t-1} + b) $$

> 이하 LSTM 수식에서 $\sigma$ 는 시그모이드 함수, $\tanh$ 는 하이퍼볼릭탄젠트 함수를 뜻한다. 게이트마다 $x_t$ 와 곱해지는 가중치 $W_{xi}, W_{xg}, W_{xf}, W_{xo}$, $h_{t-1}$ 과 곱해지는 가중치 $W_{hi}, W_{hg}, W_{hf}, W_{ho}$, 그리고 편향 $b_i, b_g, b_f, b_o$ 가 사용된다. 세 게이트에는 공통적으로 **시그모이드 함수**가 있어, 0과 1 사이의 값으로 게이트를 조절한다.

## 입력 게이트 (Input Gate)

입력 게이트는 **현재 정보를 기억하기 위한** 게이트다. 현재 시점 $t$ 의 입력 $x_t$ 와 이전 은닉 상태 $h_{t-1}$ 를 각각의 가중치와 곱해 더한 뒤, 한쪽은 시그모이드를, 다른 한쪽은 하이퍼볼릭탄젠트를 통과시킨다.

$$ i_t = \sigma(W_{xi} x_t + W_{hi} h_{t-1} + b_i) $$
$$ g_t = \tanh(W_{xg} x_t + W_{hg} h_{t-1} + b_g) $$

시그모이드를 지난 0~1 사이의 값 $i_t$ 와 하이퍼볼릭탄젠트를 지난 −1~1 사이의 값 $g_t$, 이 두 값으로 이번에 기억할 정보의 양을 정한다. 구체적인 결합 방식은 아래 셀 상태 식에서 확인한다.

## 삭제 게이트 (Forget Gate)

삭제 게이트는 **기억을 삭제하기 위한** 게이트다. 현재 입력 $x_t$ 와 이전 은닉 상태 $h_{t-1}$ 를 시그모이드 함수에 통과시킨다.

$$ f_t = \sigma(W_{xf} x_t + W_{hf} h_{t-1} + b_f) $$

시그모이드를 지난 0~1 사이의 값 $f_t$ 가 곧 삭제 과정을 거치고 **남은 정보의 양**이다. 0에 가까울수록 많이 삭제된 것이고, 1에 가까울수록 정보를 온전히 기억한 것이다.

## 셀 상태 (Cell State, 장기 상태)

셀 상태 $C_t$ 는 LSTM의 **장기 상태**라고도 부른다. 입력 게이트에서 구한 $i_t, g_t$ 를 원소별 곱(entrywise product, $\circ$)한 값이 이번에 새로 기억할 값이고, 삭제 게이트의 결과로 걸러진 이전 셀 상태와 더해 현재 셀 상태를 만든다.

$$ C_t = f_t \circ C_{t-1} + i_t \circ g_t $$

> **삭제 게이트와 입력 게이트의 영향력**
> - 삭제 게이트 $f_t = 0$ 이면 이전 셀 상태 $C_{t-1}$ 의 영향력이 0이 되어, 오직 입력 게이트의 결과만으로 현재 셀 상태가 결정된다.
> - 입력 게이트 $i_t = 0$ 이면 현재 셀 상태는 오직 이전 셀 상태 $C_{t-1}$ 에만 의존한다.
>
> 결국 **삭제 게이트는 이전 시점의 입력을 얼마나 반영할지**, **입력 게이트는 현재 시점의 입력을 얼마나 반영할지**를 결정한다.

## 출력 게이트와 은닉 상태 (단기 상태)

출력 게이트는 현재 입력 $x_t$ 와 이전 은닉 상태 $h_{t-1}$ 를 시그모이드에 통과시킨 값으로, 현재 시점의 은닉 상태를 결정하는 데 쓰인다.

$$ o_t = \sigma(W_{xo} x_t + W_{ho} h_{t-1} + b_o) $$
$$ h_t = o_t \circ \tanh(C_t) $$

은닉 상태는 **단기 상태**라고도 한다. 장기 상태(셀 상태)가 하이퍼볼릭탄젠트를 지나 −1~1 사이의 값이 되고, 여기에 출력 게이트 값이 곱해져 값이 걸러진다. 이 단기 상태값은 다음 시점으로 전달되는 동시에 출력층으로도 향한다.

## 파이토치의 nn.LSTM()

파이토치에서 LSTM 셀을 쓰는 방법은 RNN과 거의 같다. 기존 RNN 셀은 `nn.RNN(input_dim, hidden_size, batch_first=True)` 처럼 썼는데, LSTM 셀은 이름만 바꿔 `nn.LSTM(...)`으로 쓴다.

In [ ]:
import torch.nn as nn

input_dim = 5
hidden_size = 8

# 기존 RNN 셀
rnn_cell = nn.RNN(input_dim, hidden_size, batch_first=True)

# LSTM 셀은 이름만 바꿔서 동일하게 사용
lstm_cell = nn.LSTM(input_dim, hidden_size, batch_first=True)

print(rnn_cell)
print(lstm_cell)

# 8. GRU (Gated Recurrent Unit)

**GRU(Gated Recurrent Unit)**는 2014년 뉴욕대학교 조경현 교수가 제안한 모델로, LSTM의 장기 의존성 해결책은 유지하면서 은닉 상태를 업데이트하는 계산을 줄여 구조를 단순화했다. LSTM에 3개의 게이트가 있었던 것과 달리, GRU에는 **업데이트 게이트**와 **리셋 게이트** 두 가지만 존재한다.

$$ r_t = \sigma(W_{xr} x_t + W_{hr} h_{t-1} + b_r) $$
$$ z_t = \sigma(W_{xz} x_t + W_{hz} h_{t-1} + b_z) $$
$$ g_t = \tanh(W_{hg}(r_t \circ h_{t-1}) + W_{xg} x_t + b_g) $$
$$ h_t = (1 - z_t) \circ g_t + z_t \circ h_{t-1} $$

> GRU와 LSTM 중 무엇이 더 낫다고 단정할 수는 없다. 이미 LSTM으로 최적의 하이퍼파라미터를 찾았다면 굳이 바꿀 필요는 없다. 경험적으로 **데이터가 적을 때는 매개변수가 적은 GRU**가, **데이터가 많을 때는 LSTM**이 조금 더 낫다고 알려져 있다. 다만 LSTM이 먼저 나온 구조라 연구·사용량은 LSTM이 더 많다.

## 파이토치의 nn.GRU()

GRU 셀 역시 RNN·LSTM과 동일한 방식으로 쓴다. `nn.RNN(...)` 자리에 `nn.GRU(...)`를 넣으면 된다.

In [ ]:
input_dim = 5
hidden_size = 8

# 기존 RNN 셀
rnn_cell = nn.RNN(input_dim, hidden_size, batch_first=True)

# GRU 셀은 이름만 바꿔서 동일하게 사용
gru_cell = nn.GRU(input_dim, hidden_size, batch_first=True)

print(rnn_cell)
print(gru_cell)

# 9. 문자 단위 RNN (Char RNN)

RNN의 입출력 단위를 단어 레벨(word-level)이 아니라 **문자 레벨(character-level)**로 하면 이를 **문자 단위 RNN(Char RNN)**이라 한다. RNN 구조 자체가 달라지는 것은 아니고, 입출력의 단위만 문자로 바뀐 것이다. 여기서는 문자 시퀀스 `apple`을 입력받으면 `pple!`을 출력하는 다대다(many-to-many) RNN을 구현한다.

> 이 예제 자체에 큰 의미가 있는 것은 아니고, 순수하게 RNN의 동작을 이해하기 위한 목적이다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

## 1) 훈련 데이터 전처리

입력·레이블 데이터에 등장하는 문자들의 중복을 제거해 **문자 집합(vocabulary)**을 만든다. 입력은 원-핫 벡터를 쓸 것이므로 입력의 크기는 문자 집합의 크기와 같아야 한다.

In [ ]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str + label_str)))
vocab_size = len(char_vocab)
print('문자 집합의 크기 : {}'.format(vocab_size))

현재 문자 집합에는 `!, a, e, l, p` 총 5개의 문자가 있다. 입력은 원-핫 벡터를 사용하므로 입력의 크기는 문자 집합의 크기여야 한다. 하이퍼파라미터를 정의한다.

In [ ]:
input_size = vocab_size  # 입력의 크기는 문자 집합의 크기
hidden_size = 5
output_size = 5
learning_rate = 0.1

문자 집합의 각 문자에 고유한 정수를 부여한다. 나중에 예측 결과를 다시 문자 시퀀스로 보기 위해, 반대로 정수로부터 문자를 얻는 `index_to_char` 도 만든다.

In [ ]:
# 문자에 고유한 정수 인덱스 부여
char_to_index = dict((c, i) for i, c in enumerate(char_vocab))
print(char_to_index)

In [ ]:
# 정수로부터 문자를 얻는 사전
index_to_char = {}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

입력 데이터와 레이블 데이터의 각 문자를 정수로 맵핑한다.

In [ ]:
x_data = [char_to_index[c] for c in input_str]  # a, p, p, l, e
y_data = [char_to_index[c] for c in label_str]  # p, p, l, e, !
print(x_data)
print(y_data)

파이토치의 `nn.RNN()`은 기본적으로 3차원 텐서를 입력받으므로 배치 차원을 추가한다.

In [ ]:
# 배치 차원 추가 (unsqueeze(0)로도 가능)
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

입력 시퀀스의 각 문자를 원-핫 벡터로 바꾼다.

In [ ]:
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

입력 데이터와 레이블 데이터를 텐서로 바꾸고 크기를 확인한다.

In [ ]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터는 `(1, 5, 5)`, 레이블은 `(1, 5)` 크기를 가진다. 각각 (배치, 시점, 입력 크기)와 (배치, 시점)이다.

## 2) 모델 구현

RNN 모델을 정의한다. `fc`는 완전 연결층(fully-connected layer)으로 **출력층** 역할을 한다.

In [ ]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True)  # RNN 셀
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True)      # 출력층

    def forward(self, x):
        x, _status = self.rnn(x)  # RNN 셀과
        x = self.fc(x)            # 출력층을 연결
        return x

클래스로 정의한 모델을 `net`에 저장하고, 입력을 넣어 출력의 크기를 확인한다.

In [ ]:
net = Net(input_size, hidden_size, output_size)
outputs = net(X)
print(outputs.shape)  # 3차원 텐서 (배치, 시점, 출력 크기)

출력은 `(1, 5, 5)` 크기다. 나중에 정확도를 측정할 때는 `view`를 사용해 배치 차원과 시점 차원을 하나로 합쳐 펼친다.

In [ ]:
print(outputs.view(-1, input_size).shape)  # 2차원 텐서로 변환

차원이 `(5, 5)`가 되었다. 레이블 데이터도 마찬가지로 펼친다.

In [ ]:
print(Y.shape)
print(Y.view(-1).shape)

레이블은 `(1, 5)` → `(5)`가 된다. 이제 옵티마이저와 손실 함수를 정의한다.

> **주의** : `nn.CrossEntropyLoss`는 내부에 소프트맥스가 포함되어 있다. 따라서 모델의 출력(logits)에 소프트맥스를 다시 적용하지 않고, 그대로 손실 함수에 넘긴다.

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

## 3) 학습

총 100번의 에포크를 학습한다. `view`로 배치 차원을 제거해 손실을 계산하고, 매 에포크마다 모델이 실제 어떻게 예측했는지 문자열로 복원해 확인한다.

In [ ]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    # view로 (배치 x 시점, 클래스) 형태로 펼쳐서 손실 계산
    loss = criterion(outputs.view(-1, input_size), Y.view(-1))
    loss.backward()   # 기울기 계산
    optimizer.step()  # 가중치 갱신

    # 아래 세 줄은 모델이 실제 어떻게 예측했는지 확인하기 위한 코드
    result = outputs.data.numpy().argmax(axis=2)  # 각 시점에서 가장 높은 값의 인덱스 선택
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss:", loss.item(), "prediction:", result,
          "true Y:", y_data, "prediction str:", result_str)

처음에는 엉뚱하게 예측하지만, 마지막 에포크에서는 목표였던 `pple!`을 정확히 생성한다.

# 10. 더 많은 데이터로 학습한 문자 단위 RNN

이번에는 더 긴 문장을 가지고 문자 단위 RNN을 구현한다. 긴 문장을 `sequence_length` 단위로 잘라 여러 개의 샘플을 만들고, 은닉층을 2개 쌓아 학습한다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

## 1) 훈련 데이터 전처리

문자 집합을 생성하고 각 문자에 고유한 정수를 부여한다. 공백도 여기서는 하나의 원소가 된다.

In [ ]:
char_set = list(set(sentence))                    # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩
print(char_dic)

문자 집합의 크기를 확인한다.

In [ ]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기는 25이며, 입력을 원-핫 벡터로 쓸 것이므로 이는 매 시점 입력의 크기이기도 하다. 하이퍼파라미터를 설정한다. `hidden_size`를 입력의 크기와 동일하게 줬는데, 이는 사용자의 선택으로 다른 값을 줘도 무방하다. `sequence_length`는 긴 문장을 자르는 단위다.

In [ ]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10  # 임의의 숫자 지정
learning_rate = 0.1

`sequence_length` 값인 10 단위로 샘플을 잘라 데이터를 만든다. 입력은 한 글자씩, 레이블은 그보다 한 칸 쉬프트된 시퀀스다.

In [ ]:
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1:i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index

한 칸씩 쉬프트된 시퀀스가 정상적으로 출력된다. 총 170개의 샘플이 생성되며, 첫 번째 샘플의 입력·레이블을 출력해 본다.

In [ ]:
print(x_data[0])  # if you wan 에 해당
print(y_data[0])  # f you want 에 해당

입력 시퀀스에 원-핫 인코딩을 수행하고, 입력·레이블 데이터를 텐서로 변환한다.

In [ ]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data]  # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터는 `(170, 10, 25)`, 레이블은 `(170, 10)` 크기를 가진다. 원-핫 인코딩 결과를 보기 위해 첫 번째 샘플만 출력한다.

In [ ]:
print(X[0])

레이블 데이터의 첫 번째 샘플도 출력한다.

In [ ]:
print(Y[0])

## 2) 모델 구현

모델은 앞 실습과 거의 동일하되, 이번에는 은닉층을 **두 개** 쌓는다. `nn.RNN()`의 `num_layers` 인자에 2를 전달한다.

In [ ]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers):  # hidden_dim은 dic_size와 같음
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [ ]:
net = Net(dic_size, hidden_size, 2)  # 이번에는 층을 두 개 쌓는다

비용 함수와 옵티마이저를 선언한다.

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

모델에 입력을 넣어 출력의 크기를 확인한다.

In [ ]:
outputs = net(X)
print(outputs.shape)  # 3차원 텐서 (170, 10, 25)

출력은 `(170, 10, 25)` 크기다. 정확도를 측정할 때는 `view`로 배치·시점 차원을 합쳐 펼친다.

In [ ]:
print(outputs.view(-1, dic_size).shape)  # 2차원 텐서로 변환

차원이 `(1700, 25)`가 되었다. 레이블 데이터도 마찬가지로 펼친다.

In [ ]:
print(Y.shape)
print(Y.view(-1).shape)

레이블은 `(170, 10)` → `(1700)`이 된다.

## 3) 학습

100번의 에포크를 학습한다. 매 에포크마다 예측 결과를 이어 붙여 문자열로 복원한다. 첫 샘플은 예측 결과를 전부 가져오고, 그 다음부터는 각 샘플의 **마지막 글자만** 이어 붙인다.

In [ ]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)  # (170, 10, 25) 크기를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0:  # 처음에는 예측 결과를 전부 가져온다
            predict_str += ''.join([char_set[t] for t in result])
        else:       # 그 다음에는 마지막 글자만 반복해서 추가한다
            predict_str += char_set[result[-1]]

    print(predict_str)

처음에는 이상한 예측을 하지만, 마지막 에포크에서는 원래 문장에 가까운 문자열을 꽤 정확하게 생성하는 것을 볼 수 있다.

# 정리

이번 장에서 다룬 내용을 한눈에 정리하면 다음과 같다.

- **RNN** : 은닉 상태를 다음 시점으로 되먹임하는 시퀀스 모델. $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$ 로 계산하며, 가중치는 모든 시점에서 공유된다.
- **nn.RNN()** : 두 개의 값(모든 시점의 은닉 상태, 마지막 시점의 은닉 상태)을 리턴한다. `num_layers`로 깊게, `bidirectional=True`로 양방향으로 확장한다.
- **장기 의존성 문제** : 바닐라 RNN은 시점이 길어지면 앞쪽 정보가 손실된다.
- **LSTM** : 셀 상태와 입력·삭제·출력 게이트를 도입해 장기 의존성을 보완했다.
- **GRU** : 업데이트·리셋 두 게이트로 LSTM을 단순화한 모델.
- **Char RNN** : 입출력 단위를 문자로 둔 RNN. `nn.CrossEntropyLoss`에 소프트맥스가 포함되어 있으므로 출력(logits)을 그대로 손실 함수에 넘긴다.

> 다음 장에서는 합성곱 신경망(Convolutional Neural Network, CNN)을 다룬다.